# DataHub 接入与验证（开发脚本）

> **使用对象**：要修改 DataHub 接入流程 / 验证数据上报的开发者
> **不推荐**：给「小白」学习用 —— 这是 OpenSearch / GraphQL 的硬核 API，
> 不在「数据治理入门」必修路径上。教学路径请回到 [`step1.ipynb`](./step1.ipynb)。

## 这个 notebook 做什么

把 5 个系统的元数据从本地 `data/historical/` 通过 `scripts/direct_es_bulk.py` 写入
DataHub 后端的 OpenSearch 索引，并验证：

1. 各服务健康状态（GMS / OpenSearch / Neo4j）
2. 清除 OpenSearch 中已有的资产（模拟「重新上报」）
3. 调用 `direct_es_bulk.py` 重新写入
4. 验证：OpenSearch `_count` / `_search`
5. 验证：DataHub Browse GraphQL
6. 验证：DataHub Search GraphQL

**前置条件**：DataHub 已启动（`docs/Deps.md` 的 quickstart 步骤），且 `http://localhost:29002` 可达。

## 1. 服务状态确认

在重新上报前，先确认 DataHub 各服务正常运行：

In [ ]:
import requests

GMS_URL = "http://localhost:28080"
ES_URL  = "http://localhost:29200"
AUTH    = ("datahub", "datahub")

def check_service(url, name):
    try:
        r = requests.get(url, timeout=5)
        return r.status_code < 500
    except Exception:
        return False

print("=== DataHub 服务状态 ===")
print(f"  GMS API (localhost:28080):    {'OK 可达' if check_service(f'{GMS_URL}/health', 'GMS') else 'X 不可达'}")
print(f"  OpenSearch (localhost:29200): {'OK 可达' if check_service(f'{ES_URL}/_cluster/health', 'ES') else 'X 不可达'}")
print()
print("各服务端口对照：")
print("  前端 Web UI:  http://localhost:29002")
print("  GMS API:     http://localhost:28080")
print("  Neo4j:       http://localhost:27474")
print("  OpenSearch:  http://localhost:29200")

## 2. 清除已有数据

先删除 OpenSearch 中已存在的资产数据，模拟「重新上报」场景。
**警告**：这是 `delete_by_query` —— 会清掉符合 URN 模式的所有资产，请确认不要影响生产数据。

In [ ]:
def es_count(index):
    r = requests.get(f"{ES_URL}/{index}/_count", timeout=10)
    return r.json().get('count', 0)

print("=== 删除前 ===")
cnt_before = es_count('datasetindex_v2')
print(f"datasetindex_v2 文档数: {cnt_before}")

delete_by_query = {
    "query": {
        "bool": {
            "should": [
                {"wildcard": {"urn": "urn:li:dataset:(urn:li:dataPlatform:lims,*"}},
                {"wildcard": {"urn": "urn:li:dataset:(urn:li:dataPlatform:sap_erp,*"}},
                {"wildcard": {"urn": "urn:li:dataset:(urn:li:dataPlatform:pi_system,*"}},
                {"wildcard": {"urn": "urn:li:dataset:(urn:li:dataPlatform:oa,*"}},
                {"wildcard": {"urn": "urn:li:dataset:(urn:li:dataPlatform:scada,*"}},
            ],
            "minimum_should_match": 1
        }
    }
}
r = requests.post(
    f"{ES_URL}/datasetindex_v2/_delete_by_query",
    json=delete_by_query,
    headers={"Content-Type": "application/json"},
    timeout=30,
)
result = r.json()
deleted = result.get('deleted', 0)
print(f"已删除 {deleted} 条文档")

print("=== 删除后 ===")
cnt_after = es_count('datasetindex_v2')
print(f"datasetindex_v2 文档数: {cnt_after}")

## 3. 重新导入数据

调用 `scripts/direct_es_bulk.py` 将资产重新写入 OpenSearch。
这个脚本读取 `data/historical/` 下的元数据 + `src/dg_platform/datahub_client.py` 的映射规则，
生成符合 DataHub 期望的 MCE 事件 JSON，再通过 OpenSearch `_bulk` 端点直接写入（绕开 GMS）。

In [ ]:
import subprocess, os

WORK_DIR = "/home/szs/Playground/dg-demo"
script_path = os.path.join(WORK_DIR, "scripts", "direct_es_bulk.py")
print(f"执行脚本: {script_path}")
print("输出:")
result = subprocess.run(
    ["uv", "run", "python", script_path],
    cwd=WORK_DIR,
    capture_output=True,
    text=True,
    timeout=120,
)
print(result.stdout[-2000:] if len(result.stdout) > 2000 else result.stdout)
if result.stderr:
    print("STDERR:", result.stderr[-500:])
print(f"\n返回码: {result.returncode}")

## 4. 验证：导入后数据

通过 OpenSearch REST API 直接查询 dataset index，确认资产是否已录入。

In [ ]:
r = requests.get(f"{ES_URL}/datasetindex_v2/_count", timeout=10)
d = r.json()
total = d.get('count', 0)
print(f"OpenSearch datasetindex_v2 中资产数量: {total}")
print()

# 按平台统计
r2 = requests.get(f"{ES_URL}/datasetindex_v2/_search?size=50&_source=platform,name,urn", timeout=10)
d2 = r2.json()

platform_count = {}
assets = []
for h in d2.get('hits', {}).get('hits', []):
    s = h['_source']
    plat = s.get('platform', 'unknown')
    platform_count[plat] = platform_count.get(plat, 0) + 1
    assets.append({'platform': plat, 'name': s.get('name', ''), 'urn': s.get('urn', '')})

print("=== 各平台资产统计 ===")
for plat, cnt in sorted(platform_count.items()):
    print(f"  {plat:15s}: {cnt} 张表")
print()
print("=== 资产清单 ===")
for a in sorted(assets, key=lambda x: (x['platform'], x['name'])):
    print(f"  {a['platform']:15s} / {a['name']:20s}")

## 5. 验证：DataHub Browse 页面

通过 GMS GraphQL API 查询 Browse 侧边栏，确认左侧导航能展示各平台。

In [ ]:
platforms = ["lims", "sap_erp", "pi_system", "oa"]
print("=== DataHub Browse API 验证 ===")
for platform in platforms:
    browse_q = {
        "query": f'''
        query BrowsePlatform {{
            browse(input: {{ type: DATASET, path: ["{platform}"], start: 0, count: 10 }}) {{
                total
                groups {{ name count }}
            }}
        }}
        '''
    }
    try:
        r = requests.post(
            f"{GMS_URL}/api/graphql",
            json=browse_q,
            auth=AUTH,
            timeout=10,
        )
        d = r.json()
        errors = d.get("errors", [])
        if errors:
            print(f"  {platform}: 查询失败 — {errors[0]['message'][:80]}")
            continue
        browse = d.get("data", {}).get("browse", {})
        total = browse.get("total", 0)
        groups = browse.get("groups", [])
        print(f"  {platform}: {total} 张表")
        for g in groups:
            print(f"    └─ {g['name']}: {g['count']}")
    except Exception as e:
        print(f"  {platform}: 请求异常 — {e}")

## 6. 验证：搜索功能

通过 GraphQL `searchAcrossEntities` 验证搜索是否正常工作：

In [ ]:
search_q = {
    "query": '''
    query SearchVerify {{
        searchAcrossEntities(input: {{ query: "*", type: DATASET, start: 0, count: 20 }}) {{
            total
            searchResults {{
                entity {{
                    ... on Dataset {{ name platform {{ name }} }}
                }}
            }}
        }}
    }}
    '''
}
try:
    r = requests.post(f"{GMS_URL}/api/graphql", json=search_q, auth=AUTH, timeout=10)
    d = r.json()
    errors = d.get("errors", [])
    if errors:
        print(f"搜索失败: {errors[0]['message'][:100]}")
    else:
        results = d.get("data", {}).get("searchAcrossEntities", {})
        total = results.get("total", 0)
        items = results.get("searchResults", [])
        print(f"=== 搜索结果: 共 {total} 条匹配 (显示前 {len(items)} 条) ===")
        for res in items:
            ent = res.get("entity", {})
            print(f"  {ent.get('platform',{}).get('name','?')}/{ent.get('name','?')}")
except Exception as e:
    print(f"搜索请求异常: {e}")

## 7. 结论

本 notebook 演示了完整的数据资产上报流程：

```
1. 清除已有数据  →  OpenSearch DELETE by query
2. 重新导入数据  →  scripts/direct_es_bulk.py
3. 验证写入结果  →  OpenSearch _count + _search
4. 验证Browse导航 →  GMS GraphQL browse API
5. 验证搜索功能  →  GMS GraphQL searchAcrossEntities
```

**关键操作**：
- 写入 API：`POST /_bulk` 直接 bulk 写入 OpenSearch（绕开 GMS，走 ingestion pipe）
- 验证 API：`POST /datasetindex_v2/_search` 确认数据存在
- Browse API：`POST /api/graphql` GraphQL browse 查询导航路径

> 若需通过 GMS 正式上报（写入 MySQL + 同步 OpenSearch），应使用 `datahub` CLI 的 `ingest` 命令，
> 或调用 `POST /operations?action=restoreIndices` 将 MySQL 数据同步到 OpenSearch。